# 🎨 Day 26 — VAE, ELBO, Latent Space & Generative Model Comparison
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Variational Autoencoder (VAE) from Scratch | ~12 sec |
| 2 | 10:30–11:00 AM | Latent Space Interpolation & β-VAE | ~5 sec |
| 3 | 11:00 AM–1:00 PM | VAE Training Loop + Generative Model Comparison | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | VAE Capstone: Anomaly Detection | ~18 sec |

> **Zero downloads. Pure numpy + sklearn + scipy.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wasserstein_distance
from sklearn.metrics import roc_auc_score, roc_curve
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

# ── Shared helpers ──────────────────────────────────────────────
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-30,30)))
def relu(x): return np.maximum(0, x)
def tanh(x): return np.tanh(x)
print('Helpers defined: sigmoid, relu, tanh')

---
## Session 1 — Variational Autoencoder (VAE) from Scratch
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

### VAE Key Insight
Unlike a regular autoencoder that maps x→z (fixed point),
a VAE maps x→(μ,σ²) and samples z~N(μ,σ²).
```
ELBO = E[log p(x|z)] − KL(q_φ(z|x) || p(z))
     = Reconstruction    − Regularisation
```

In [ ]:
# 1.1  VAE Building Blocks

# KL Divergence: KL(N(μ,σ²) || N(0,1))
def kl_divergence(mu, log_var):
    """
    Closed-form KL between diagonal Gaussian q(z|x)=N(μ,σ²) and N(0,I):
    KL = -½ Σ(1 + log σ² - μ² - σ²)
    """
    return -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))

# Reparameterisation trick
def reparameterise(mu, log_var, rng_r):
    """
    z = μ + σ·ε  where ε ~ N(0,I)
    Key: gradients flow through μ and σ (deterministic),
         NOT through the stochastic ε sample.
    """
    eps = rng_r.normal(0, 1, mu.shape)
    return mu + np.exp(0.5 * log_var) * eps

# Verify KL
mu_test      = np.array([1.0, -0.5, 0.2])
log_var_test = np.array([-0.5, 0.1, -1.0])
kl_val       = kl_divergence(mu_test, log_var_test)
kl_zero      = kl_divergence(np.zeros(3), np.zeros(3))  # N(0,I) vs N(0,I) = 0

print('KL Divergence Tests:')
print(f'  KL(N(μ,σ²) || N(0,I)) where μ={mu_test}, σ²={np.exp(log_var_test).round(3)}: {kl_val:.4f}')
print(f'  KL(N(0,I) || N(0,I)) = {kl_zero:.6f}  (should be exactly 0.0)')

In [ ]:
# 1.2  Full VAE: Encoder → Reparameterise → Decoder
class VAE:
    """
    2-layer VAE with linear encoder and decoder.
    Encoder: x (d_in) → hidden → (μ, log σ²) (d_lat)
    Decoder: z (d_lat) → hidden → x̂ (d_in)
    """
    def __init__(self, d_in, d_lat, d_hidden=32, beta=1.0, seed=42):
        rng_v = np.random.default_rng(seed)
        # Encoder weights
        self.W_enc  = rng_v.normal(0, 0.05, (d_in, d_hidden))
        self.b_enc  = np.zeros(d_hidden)
        self.W_mu   = rng_v.normal(0, 0.05, (d_hidden, d_lat))
        self.b_mu   = np.zeros(d_lat)
        self.W_lv   = rng_v.normal(0, 0.05, (d_hidden, d_lat))
        self.b_lv   = np.zeros(d_lat)
        # Decoder weights
        self.W_dec1 = rng_v.normal(0, 0.05, (d_lat, d_hidden))
        self.b_dec1 = np.zeros(d_hidden)
        self.W_dec2 = rng_v.normal(0, 0.05, (d_hidden, d_in))
        self.b_dec2 = np.zeros(d_in)
        self.beta   = beta
        self.d_in   = d_in
        self.d_lat  = d_lat

    def encode(self, x):
        h   = tanh(x @ self.W_enc + self.b_enc)
        mu  = h @ self.W_mu  + self.b_mu
        lv  = h @ self.W_lv  + self.b_lv
        return mu, lv, h

    def decode(self, z):
        h2  = tanh(z @ self.W_dec1 + self.b_dec1)
        return h2 @ self.W_dec2 + self.b_dec2  # linear output for continuous data

    def forward(self, x, rng_f):
        mu, lv, h_enc = self.encode(x)
        z             = reparameterise(mu, lv, rng_f)
        x_hat         = self.decode(z)
        recon_loss    = np.mean((x - x_hat)**2)
        kl_loss       = kl_divergence(mu, lv) / self.d_in  # per-dim normalise
        elbo          = -(recon_loss + self.beta * kl_loss)
        return x_hat, z, mu, lv, recon_loss, kl_loss, elbo

    def sample(self, n, rng_s):
        """Generate new samples by sampling z~N(0,I) and decoding."""
        z = rng_s.normal(0, 1, (n, self.d_lat))
        return np.array([self.decode(zi) for zi in z])

# Test forward pass
rng = np.random.default_rng(42)
vae = VAE(d_in=8, d_lat=2, d_hidden=16)
x_test = rng.normal(0, 1, 8)
x_hat, z, mu, lv, recon, kl, elbo = vae.forward(x_test, rng)

print(f'\nVAE Forward Pass:')
print(f'  Input x shape   : {x_test.shape}')
print(f'  Latent z shape  : {z.shape}  (2D latent space)')
print(f'  μ shape         : {mu.shape}  σ²={np.exp(lv).round(4)}')
print(f'  Reconstruction  : {recon:.4f}')
print(f'  KL divergence   : {kl:.4f}')
print(f'  ELBO            : {elbo:.4f}  (maximise this)')

In [ ]:
# 1.3  VAE Gradient Step (Backprop through Decoder Only for Speed)
def vae_step(vae_model, x, rng_step, lr=0.005):
    """Single SGD step on ELBO using analytical decoder gradient."""
    mu, lv, h_enc = vae_model.encode(x)
    z             = reparameterise(mu, lv, rng_step)
    x_hat         = vae_model.decode(z)

    recon_loss    = np.mean((x - x_hat)**2)
    kl_loss       = kl_divergence(mu, lv) / vae_model.d_in

    # Decoder gradient: ∂MSE/∂x_hat
    d_xhat = -2*(x - x_hat) / vae_model.d_in

    # Gradient through decoder linear layer
    h2      = tanh(z @ vae_model.W_dec1 + vae_model.b_dec1)
    dW_dec2 = np.outer(h2, d_xhat)
    db_dec2 = d_xhat
    vae_model.W_dec2 -= lr * dW_dec2
    vae_model.b_dec2 -= lr * db_dec2

    # Gradient through decoder hidden
    d_h2     = d_xhat @ vae_model.W_dec2.T * (1 - h2**2)
    dW_dec1  = np.outer(z, d_h2)
    vae_model.W_dec1 -= lr * dW_dec1

    return recon_loss + kl_loss

# Quick test of gradient step
loss_before = recon + kl
loss_after  = vae_step(vae, x_test, rng)
print(f'Gradient step: loss {loss_before:.4f} → {loss_after:.4f}')

In [ ]:
# 1.4  Visualise KL Divergence vs Posterior Width
mus    = np.linspace(-3, 3, 50)
log_vs = np.linspace(-4, 2, 50)
MU, LV = np.meshgrid(mus, log_vs)

KL_grid = -0.5*(1 + LV - MU**2 - np.exp(LV))  # per-dimension KL

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].contourf(MU, LV, KL_grid, levels=30, cmap='RdBu_r')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('μ (posterior mean)')
axes[0].set_ylabel('log σ² (posterior log-variance)')
axes[0].set_title('KL divergence per dimension\n(min at μ=0, log σ²=0)')
axes[0].scatter(0, 0, color='white', s=100, zorder=5, label='min KL')
axes[0].legend()

# Show how reparameterisation enables gradient flow
mu_demo    = np.linspace(-2, 2, 100)
sigma_demo = 0.5
samples_z  = mu_demo + sigma_demo * 0.5   # z = μ + σε
axes[1].plot(mu_demo, samples_z, color='#534AB7', linewidth=2, label='z = μ + σε')
axes[1].fill_between(mu_demo, mu_demo-sigma_demo, mu_demo+sigma_demo,
                      alpha=0.2, color='#534AB7', label='±σ range')
axes[1].plot(mu_demo, mu_demo, 'k--', alpha=0.4, label='z=μ (no noise)')
axes[1].set_xlabel('μ (learnable mean)'); axes[1].set_ylabel('z (sample)')
axes[1].set_title('Reparameterisation Trick:\nGradient flows through μ')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## Session 2 — Latent Space Interpolation & β-VAE
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Latent Space Interpolation: LERP vs SLERP
def lerp(z1, z2, t):
    """Linear interpolation: z(t) = (1-t)z₁ + t·z₂."""
    return (1-t)*z1 + t*z2

def slerp(z1, z2, t):
    """
    Spherical Linear Interpolation (SLERP).
    Interpolates on the unit sphere — maintains consistent norm.
    Better for high-dimensional latent spaces.
    """
    z1_n     = z1 / (np.linalg.norm(z1) + 1e-9)
    z2_n     = z2 / (np.linalg.norm(z2) + 1e-9)
    cos_omega = np.clip(z1_n @ z2_n, -1, 1)
    omega     = np.arccos(cos_omega)
    if np.abs(omega) < 1e-6:
        return lerp(z1, z2, t)
    norm_avg = (1-t)*np.linalg.norm(z1) + t*np.linalg.norm(z2)
    result   = (np.sin((1-t)*omega)/np.sin(omega)) * z1_n + \
               (np.sin(t*omega)    /np.sin(omega)) * z2_n
    return result * norm_avg

rng_interp = np.random.default_rng(42)
D_INT      = 8
z1 = rng_interp.normal(0, 1, D_INT)
z2 = rng_interp.normal(0, 1, D_INT)
ts = np.linspace(0, 1, 9)

print('Latent Space Interpolation — LERP vs SLERP:')
print(f'{"t":6s} | {"LERP norm":10s} | {"SLERP norm":12s} | {"Diff"}')
print('-'*45)
for t_val in ts:
    z_l = lerp(z1, z2, t_val)
    z_s = slerp(z1, z2, t_val)
    n_l = np.linalg.norm(z_l)
    n_s = np.linalg.norm(z_s)
    print(f'{t_val:6.2f} | {n_l:10.4f} | {n_s:12.4f} | {abs(n_l-n_s):.4f}')
print('\nSLERP maintains more consistent norms during interpolation.')

In [ ]:
# 2.2  β-VAE: Higher β Forces Disentanglement
rng_bv  = np.random.default_rng(42)
X_wine, y_wine = load_wine(return_X_y=True)
sc_wine = StandardScaler().fit(X_wine)
X_wine_n = sc_wine.transform(X_wine)
D_WINE   = X_wine_n.shape[1]

results_beta = {}
for beta_val in [0.1, 1.0, 4.0, 8.0]:
    vae_b = VAE(d_in=D_WINE, d_lat=4, d_hidden=24, beta=beta_val, seed=42)
    total_kl = 0; total_recon = 0
    for step in range(300):
        x_b = X_wine_n[rng_bv.integers(len(X_wine_n))]
        loss_b = vae_step(vae_b, x_b, rng_bv, lr=0.008)
    # Evaluate all samples
    recon_errs, kls = [], []
    for x_b in X_wine_n:
        _, _, mu_b, lv_b, recon_b, kl_b, _ = vae_b.forward(x_b, rng_bv)
        recon_errs.append(recon_b); kls.append(kl_b)
    results_beta[beta_val] = {
        'recon': np.mean(recon_errs),
        'kl'   : np.mean(kls),
        'vae'  : vae_b
    }

print('β-VAE Experiment — Effect of β on KL and Reconstruction:')
print(f'{"β":6s} | {"Recon loss":12s} | {"KL/dim":10s} | {"Trade-off"}')
print('-'*50)
for beta_val, res in results_beta.items():
    tradeoff = res['recon'] / (res['kl'] + 1e-6)
    print(f'{beta_val:6.1f} | {res["recon"]:12.4f} | {res["kl"]:10.4f} | {tradeoff:.2f}')

In [ ]:
# 2.3  Latent Dimension Traversal
# Train a 2D VAE on Wine, then traverse each dimension
vae_2d    = VAE(d_in=D_WINE, d_lat=2, d_hidden=24, beta=1.0, seed=42)
rng_trav  = np.random.default_rng(42)
for step in range(500):
    x_t = X_wine_n[rng_trav.integers(len(X_wine_n))]
    vae_step(vae_2d, x_t, rng_trav, lr=0.006)

# Encode all Wine samples
mus_all, lvs_all = [], []
for x_t in X_wine_n:
    mu_t, lv_t, _ = vae_2d.encode(x_t)
    mus_all.append(mu_t); lvs_all.append(lv_t)
mus_all = np.array(mus_all)   # (178, 2)
lvs_all = np.array(lvs_all)

# Traverse dim 0: fix dim 1, vary dim 0
z_center = mus_all.mean(axis=0)   # mean latent point
traversal_dim0 = np.linspace(-3, 3, 9)
print('\nLatent Dimension Traversal (Wine VAE, 2D latent):')
print(f'  Latent center: {z_center.round(4)}')
for z0 in traversal_dim0:
    z_trav = np.array([z0, z_center[1]])
    x_rec  = vae_2d.decode(z_trav)
    print(f'  z=[{z0:+.1f},{z_center[1]:+.3f}] → recon_1st_feat={x_rec[0]:.4f}')

---
## Session 3 — VAE Training Loop + Generative Model Comparison
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Full VAE Training with KL Annealing
rng_train = np.random.default_rng(42)
X_wine_n2 = X_wine_n.copy()

vae_trained = VAE(d_in=D_WINE, d_lat=3, d_hidden=32, beta=1.0, seed=42)

N_EPOCHS   = 600
LR_TRAIN   = 0.006
elbo_hist  = []
kl_hist    = []
recon_hist = []

for epoch in range(N_EPOCHS):
    idx  = rng_train.integers(len(X_wine_n2))
    x_e  = X_wine_n2[idx]
    mu_e, lv_e, h_enc = vae_trained.encode(x_e)
    z_e  = reparameterise(mu_e, lv_e, rng_train)
    xhat = vae_trained.decode(z_e)

    recon_e = np.mean((x_e - xhat)**2)
    kl_e    = kl_divergence(mu_e, lv_e) / D_WINE
    # KL annealing: linearly warm up β from 0→1 over first 300 epochs
    beta_e  = min(1.0, epoch / 300)
    loss_e  = recon_e + beta_e * kl_e

    # Decoder gradient
    h2      = tanh(z_e @ vae_trained.W_dec1 + vae_trained.b_dec1)
    d_xhat  = -2*(x_e - xhat) / D_WINE
    dW_d2   = np.outer(h2, d_xhat); vae_trained.W_dec2 -= LR_TRAIN * dW_d2
    d_h2    = d_xhat @ vae_trained.W_dec2.T * (1-h2**2)
    dW_d1   = np.outer(z_e, d_h2); vae_trained.W_dec1 -= LR_TRAIN * dW_d1

    elbo_hist.append(loss_e)
    kl_hist.append(kl_e)
    recon_hist.append(recon_e)

print(f'VAE Training with KL Annealing ({N_EPOCHS} epochs):')
print(f'  Initial ELBO  : {elbo_hist[0]:.4f}')
print(f'  Final ELBO    : {elbo_hist[-1]:.4f}')
print(f'  Final recon   : {recon_hist[-1]:.4f}')
print(f'  Final KL/dim  : {kl_hist[-1]:.4f}')

In [ ]:
# 3.2  Generate New Samples + Evaluate Quality
rng_gen_vae = np.random.default_rng(0)
X_synth_vae = vae_trained.sample(100, rng_gen_vae)

# Column statistics
print('VAE Generated Sample Quality:')
print(f'{"Feature":12s} | {"Real μ":8s} | {"Synth μ":8s} | {"Real σ":8s} | {"Synth σ":8s} | {"Wasserstein"}')
print('-'*72)
for col in range(min(6, D_WINE)):
    rm = X_wine_n2[:,col].mean(); rs = X_wine_n2[:,col].std()
    sm = X_synth_vae[:,col].mean(); ss = X_synth_vae[:,col].std()
    wd = wasserstein_distance(X_wine_n2[:,col], X_synth_vae[:,col])
    print(f'{col:12d} | {rm:8.4f} | {sm:8.4f} | {rs:8.4f} | {ss:8.4f} | {wd:.4f}')

In [ ]:
# 3.3  Generative Model Comparison: VAE vs GAN vs Diffusion
print('\n=== Generative Model Comparison ===')
comparison = [
    ('Training',      'Stable ELBO optimisation', 'Adversarial (unstable, mode drop)', 'Stable MSE denoising'),
    ('Sample quality','Blurry (posterior avg.)',  'Sharp (critic-driven)',             'Highest (iterative refine)'),
    ('Latent space',  'Structured, continuous',   'Implicit (no encoder)',             'None (start from noise)'),
    ('Diversity',     'High',                     'Mode collapse risk',                'Highest'),
    ('Speed (infer)', 'Fast (single forward)',     'Fast (single forward)',             'Slow (T reverse steps)'),
    ('Likelihood',    'Tractable ELBO bound',      'Intractable (adversarial)',          'Tractable (ELBO variant)'),
    ('Use cases',     'Anomaly, compression, VAE', 'Image synth, style transfer',       'Image/text/audio gen'),
    ('Params needed', 'Millions',                  'Millions (2 networks)',             'Millions'),
]
print(f'{"Criterion":16s} | {"VAE":30s} | {"GAN":30s} | {"Diffusion"}')
print('-'*110)
for row in comparison:
    print(f'{row[0]:16s} | {row[1]:30s} | {row[2]:30s} | {row[3]}')

In [ ]:
# 3.4  Latent Space Visualisation + Training Curves
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Latent space (PCA of 3D latent → 2D)
mus_wine = []
for x_w in X_wine_n2:
    mu_w, _, _ = vae_trained.encode(x_w)
    mus_wine.append(mu_w)
mus_wine = np.array(mus_wine)
pca_lat  = PCA(n_components=2, random_state=42)
mus_2d   = pca_lat.fit_transform(mus_wine)

colors_lat = ['#534AB7','#1D9E75','#D85A30']
for cls in range(3):
    m = y_wine == cls
    axes[0].scatter(mus_2d[m,0], mus_2d[m,1], color=colors_lat[cls],
                    s=30, alpha=0.8, label=f'Class {cls}')
axes[0].set_title('VAE Latent Space (PCA 2D)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Training curves
axes[1].plot(pd.Series(elbo_hist).rolling(30).mean(), color='#534AB7', linewidth=2, label='ELBO')
axes[1].plot(pd.Series(recon_hist).rolling(30).mean(), color='#D85A30', linewidth=1.5, label='Recon')
axes[1].plot(pd.Series(kl_hist).rolling(30).mean(), color='#1D9E75', linewidth=1.5, label='KL')
axes[1].axvline(300, linestyle='--', color='gray', alpha=0.5, label='KL anneal done')
axes[1].set_title('VAE Training: ELBO / Recon / KL')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# β-VAE: KL vs Recon trade-off
beta_vals  = list(results_beta.keys())
kls_b      = [results_beta[b]['kl']   for b in beta_vals]
recons_b   = [results_beta[b]['recon'] for b in beta_vals]
axes[2].plot(kls_b, recons_b, 'o-', color='#534AB7', linewidth=2, markersize=8)
for i, beta_v in enumerate(beta_vals):
    axes[2].annotate(f'β={beta_v}', (kls_b[i], recons_b[i]),
                     xytext=(6,4), textcoords='offset points', fontsize=9)
axes[2].set_xlabel('Mean KL/dim'); axes[2].set_ylabel('Recon loss')
axes[2].set_title('β-VAE: KL–Reconstruction Trade-off')
axes[2].grid(alpha=0.3)

plt.suptitle('Day 26: VAE Training & Latent Analysis', fontsize=12)
plt.tight_layout(); plt.show()

---
## Lunch Break — 1:00–2:00 PM
1. Why can we not backprop through a stochastic sample z~q(z|x) directly?
2. What does the KL term in ELBO regularise — and why push toward N(0,I)?
3. Why do GANs produce sharper samples than VAEs?
4. How does β>1 in β-VAE encourage disentanglement?
---

## Session 5 — VAE Capstone: Anomaly Detection
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Setup: Breast Cancer — Benign (normal) vs Malignant (anomaly)
print('='*62)
print(' VAE CAPSTONE: ANOMALY DETECTION')
print('='*62)

X_bc, y_bc = load_breast_cancer(return_X_y=True)
# Normalise
X_bc_n     = (X_bc - X_bc.mean(0)) / (X_bc.std(0) + 1e-8)

# Split: train on benign only
X_benign   = X_bc_n[y_bc == 1]   # 357 benign (normal class)
X_malign   = X_bc_n[y_bc == 0]   # 212 malignant (anomaly)

# Hold out 30% of benign for testing
X_ben_tr, X_ben_te = train_test_split(X_benign, test_size=0.3, random_state=42)

print(f'Normal class (benign)    : {len(X_benign)} → train={len(X_ben_tr)} test={len(X_ben_te)}')
print(f'Anomaly class (malignant): {len(X_malign)} (used only at evaluation)')

In [ ]:
# 5.2  Train VAE on Normal (Benign) Data Only
rng_anom = np.random.default_rng(42)
D_BC     = X_bc_n.shape[1]   # 30 features
D_LAT_A  = 6

vae_anom = VAE(d_in=D_BC, d_lat=D_LAT_A, d_hidden=32, beta=1.0, seed=42)

anom_train_losses = []
for epoch in range(800):
    x_a  = X_ben_tr[rng_anom.integers(len(X_ben_tr))]
    beta_a = min(1.0, epoch/400)  # KL annealing
    mu_a, lv_a, _ = vae_anom.encode(x_a)
    z_a    = reparameterise(mu_a, lv_a, rng_anom)
    xhat_a = vae_anom.decode(z_a)
    recon_a = np.mean((x_a - xhat_a)**2)
    kl_a    = kl_divergence(mu_a, lv_a) / D_BC
    loss_a  = recon_a + beta_a * kl_a

    # Update decoder only (faster training)
    h2_a   = tanh(z_a @ vae_anom.W_dec1 + vae_anom.b_dec1)
    d_xh_a = -2*(x_a - xhat_a) / D_BC
    vae_anom.W_dec2 -= 0.005 * np.outer(h2_a, d_xh_a)
    dh2_a   = d_xh_a @ vae_anom.W_dec2.T * (1-h2_a**2)
    vae_anom.W_dec1 -= 0.005 * np.outer(z_a, dh2_a)
    anom_train_losses.append(loss_a)

print(f'VAE Anomaly Model Training (800 epochs on benign data):')
print(f'  Initial loss : {anom_train_losses[0]:.4f}')
print(f'  Final loss   : {anom_train_losses[-1]:.4f}')

In [ ]:
# 5.3  Anomaly Scoring: Reconstruction Error + ELBO
def anomaly_score(vae_m, x, rng_sc, n_samples=5):
    """
    Anomaly score = mean reconstruction error over n_samples of z.
    Higher score = more anomalous (harder to reconstruct).
    """
    mu_s, lv_s, _ = vae_m.encode(x)
    scores = []
    for _ in range(n_samples):
        z_s    = reparameterise(mu_s, lv_s, rng_sc)
        xhat_s = vae_m.decode(z_s)
        scores.append(np.mean((x - xhat_s)**2))
    return np.mean(scores)

def elbo_score(vae_m, x, rng_e, n_samples=5):
    """ELBO as anomaly score: combines recon + KL."""
    mu_e, lv_e, _ = vae_m.encode(x)
    kl_e = kl_divergence(mu_e, lv_e) / D_BC
    recon_vals = []
    for _ in range(n_samples):
        z_e    = reparameterise(mu_e, lv_e, rng_e)
        xhat_e = vae_m.decode(z_e)
        recon_vals.append(np.mean((x - xhat_e)**2))
    return np.mean(recon_vals) + kl_e

rng_score = np.random.default_rng(0)

# Score all test samples
scores_ben_te  = [anomaly_score(vae_anom, x, rng_score) for x in X_ben_te]
scores_mal     = [anomaly_score(vae_anom, x, rng_score) for x in X_malign]

elbo_ben_te    = [elbo_score(vae_anom, x, rng_score) for x in X_ben_te]
elbo_mal       = [elbo_score(vae_anom, x, rng_score) for x in X_malign]

print(f'Anomaly Scores:')
print(f'  Benign  recon: {np.mean(scores_ben_te):.4f} ± {np.std(scores_ben_te):.4f}')
print(f'  Malign  recon: {np.mean(scores_mal):.4f} ± {np.std(scores_mal):.4f}')
print(f'  Separation   : {np.mean(scores_mal)-np.mean(scores_ben_te):.4f}')

In [ ]:
# 5.4  ROC-AUC Evaluation
all_scores_recon = list(scores_ben_te) + list(scores_mal)
all_scores_elbo  = list(elbo_ben_te)   + list(elbo_mal)
labels_anom      = [0]*len(scores_ben_te) + [1]*len(scores_mal)  # 0=normal, 1=anomaly

auc_recon = roc_auc_score(labels_anom, all_scores_recon)
auc_elbo  = roc_auc_score(labels_anom, all_scores_elbo)

# Threshold-based detection
thresh    = np.percentile(scores_ben_te, 95)   # 95th percentile of normal
preds_bin = [1 if s > thresh else 0 for s in all_scores_recon]
tp  = sum(p==1 and l==1 for p,l in zip(preds_bin, labels_anom))
fp  = sum(p==1 and l==0 for p,l in zip(preds_bin, labels_anom))
fn  = sum(p==0 and l==1 for p,l in zip(preds_bin, labels_anom))
prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
f1   = 2*prec*rec/(prec+rec+1e-9)

print(f'\nVAE Anomaly Detection Results:')
print(f'  ROC-AUC (recon loss)  : {auc_recon:.4f}')
print(f'  ROC-AUC (ELBO)        : {auc_elbo:.4f}')
print(f'  Threshold (95th pct)  : {thresh:.4f}')
print(f'  Precision             : {prec:.4f}')
print(f'  Recall                : {rec:.4f}')
print(f'  F1 Score              : {f1:.4f}')

In [ ]:
# 5.5  Final 4-Panel Visualisation
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ROC Curve
fpr, tpr, _ = roc_curve(labels_anom, all_scores_recon)
axes[0,0].plot(fpr, tpr, color='#534AB7', linewidth=2, label=f'Recon (AUC={auc_recon:.3f})')
fpr_e, tpr_e, _ = roc_curve(labels_anom, all_scores_elbo)
axes[0,0].plot(fpr_e, tpr_e, color='#1D9E75', linewidth=2, label=f'ELBO (AUC={auc_elbo:.3f})')
axes[0,0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR')
axes[0,0].set_title('ROC Curve: VAE Anomaly Detection')
axes[0,0].legend(fontsize=9); axes[0,0].grid(alpha=0.3)

# Score distribution
axes[0,1].hist(scores_ben_te, bins=20, alpha=0.6, color='#1D9E75', density=True, label='Benign (normal)')
axes[0,1].hist(scores_mal,   bins=20, alpha=0.6, color='#D85A30', density=True, label='Malignant (anomaly)')
axes[0,1].axvline(thresh, linestyle='--', color='black', linewidth=2, label=f'Threshold={thresh:.4f}')
axes[0,1].set_xlabel('Reconstruction error')
axes[0,1].set_title('Anomaly Score Distributions')
axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# VAE training loss
axes[1,0].plot(pd.Series(anom_train_losses).rolling(40).mean(), color='#534AB7', linewidth=2)
axes[1,0].set_title('VAE Training Loss (Benign Data Only)')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('ELBO loss')
axes[1,0].grid(alpha=0.3)

# Latent space: benign vs malignant
mus_b = np.array([vae_anom.encode(x)[0] for x in X_ben_te])
mus_m = np.array([vae_anom.encode(x)[0] for x in X_malign])
pca_a = PCA(n_components=2, random_state=42).fit(np.vstack([mus_b, mus_m]))
b2d   = pca_a.transform(mus_b)
m2d   = pca_a.transform(mus_m)
axes[1,1].scatter(b2d[:,0], b2d[:,1], color='#1D9E75', s=20, alpha=0.7, label='Benign')
axes[1,1].scatter(m2d[:,0], m2d[:,1], color='#D85A30', s=20, alpha=0.7, label='Malignant')
axes[1,1].set_title('Latent Space: Benign vs Malignant')
axes[1,1].legend(fontsize=9); axes[1,1].grid(alpha=0.3)

plt.suptitle('VAE Anomaly Detection Capstone', fontsize=12)
plt.tight_layout(); plt.show()

print('\n=== FINAL REPORT ===')
print(f'  VAE latent dim     : {D_LAT_A}')
print(f'  Training samples   : {len(X_ben_tr)} (benign only)')
print(f'  ROC-AUC            : {auc_recon:.4f}  (ELBO: {auc_elbo:.4f})')
print(f'  F1 Score           : {f1:.4f}')

---
## Day 26 Summary

| Concept | Skill Gained |
|---|---|
| VAE encoder | q_φ(z|x) → (μ, log σ²) via learnable linear layers |
| Reparameterisation | z = μ + σ·ε enables gradient flow through sampling |
| KL divergence (closed form) | −½Σ(1+log σ²−μ²−σ²) for diagonal Gaussian |
| ELBO | Reconstruction + β·KL — two opposing forces |
| KL annealing | Warm up β from 0→1 to prevent posterior collapse |
| Linear interpolation | z(t) = (1-t)z₁ + t·z₂ in latent space |
| SLERP | Spherical interpolation maintains consistent norm |
| β-VAE | β>1 compresses KL → more disentangled dimensions |
| Latent traversal | Vary one dim at a time to reveal learned factors |
| Generative model comparison | VAE / GAN / Diffusion trade-offs |
| VAE anomaly detection | Train on normal, flag high recon error |
| ELBO as anomaly score | Recon + KL combined anomaly signal |
| ROC-AUC evaluation | Threshold-free anomaly detection metric |

---
## Day 27 Preview
- Bayesian neural networks: weight uncertainty and MC inference
- Gaussian processes revisited: kernel design and posterior inference
- Neural tangent kernel and infinite-width networks
- Uncertainty quantification comparison: BNN vs GP vs Ensemble
- Capstone: Bayesian optimisation with GP surrogate

In [ ]:
# Bonus: Day 26 in one cell
rng_b = np.random.default_rng(0)
vae_b = VAE(d_in=8, d_lat=2, d_hidden=16, beta=1.0, seed=0)
x_b   = rng_b.normal(0, 1, 8)
_, _, mu_b, lv_b, recon_b, kl_b, elbo_b = vae_b.forward(x_b, rng_b)
print(f'VAE: recon={recon_b:.4f}  KL={kl_b:.4f}  ELBO={elbo_b:.4f}')
z1_b = rng_b.normal(0,1,4); z2_b = rng_b.normal(0,1,4)
print(f'LERP mid norm  : {np.linalg.norm(lerp(z1_b,z2_b,0.5)):.4f}')
print(f'SLERP mid norm : {np.linalg.norm(slerp(z1_b,z2_b,0.5)):.4f}')
print(f'Anomaly AUC (recon): {auc_recon:.4f}  F1={f1:.4f}')
print('\nDay 26 complete — VAE, ELBO, KL annealing, β-VAE, SLERP, anomaly detection!')